# Qwen


In [1]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch
import time
import pandas as pd

In [4]:
model = 'Qwen/Qwen2.5-0.5B'
tokenizer = AutoTokenizer.from_pretrained(model)
qwen = AutoModel.from_pretrained(model)
causal_lm = AutoModelForCausalLM.from_pretrained(model)

In [5]:
def generate_response(prompt, strategy, model, causal_lm, tokenizer, n=100, T=1, k=None, p=None, eos_id=None):
    """
    Enter a prompt to generate the next tokens in a loop until maximum token limit or end of sequence or user interruption occurs.\n
    model = GPT decoder model\n
    tokenizer = tokenizer same as the model, mixing different tokenizer might result in broken text generation\n
    strategy = ["Greedy", "Temperature", "Top-k", "Top-p"]\n
    n = number of tokens, T = temperature, k = k in top-k, p = p in top-p\n
    eos_id -> end of sequence id used to stop the text generation on reaching end of sequence. If set None the model keeps on generating text until it reaches the maximum token limit.\n
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    temperature=T
    eos = eos_id
    past_key_values = None
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    generated_ids = input_ids.clone()
    prompt_tokens = len(tokenizer(prompt).input_ids)

    if strategy=='Greedy':
        start = time.perf_counter()
        for x in range(n):
            with torch.no_grad():
                base_output = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
                hidden_states = base_output.last_hidden_state
                base_logits = causal_lm.lm_head(hidden_states)
                past_key_values = base_output.past_key_values

            scaled_logits = base_logits[:, -1, :]/temperature
            probs = torch.softmax(scaled_logits, dim=-1)

            next_token_id = torch.argmax(probs ,dim=-1, keepdim=True)
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
            if x==0:
              ttfs = time.perf_counter() - start

            if next_token_id==eos:
                break

            next_token = tokenizer.decode(next_token_id)
            input_ids = next_token_id

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
        total_time=time.perf_counter() - start

    elif strategy=="Temperature":
        start = time.perf_counter()
        for x in range(n):
            with torch.no_grad():
                base_output = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
                hidden_state = base_output.last_hidden_state
                base_logits = causal_lm.lm_head(hidden_state)
                past_key_values = base_output.past_key_values

            scaled_logits = base_logits[:, -1, :]/temperature
            probs = torch.softmax(scaled_logits, dim=-1)

            next_token_id = torch.multinomial(probs, num_samples=1)
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
            if x==0:
              ttfs = time.perf_counter() - start

            if next_token_id==eos:
                break

            input_ids = next_token_id

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
        total_time=time.perf_counter() - start

    elif strategy=="Top-k":
        start = time.perf_counter()
        for x in range(n):
            with torch.no_grad():
                bo = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
                hs = bo.last_hidden_state
                base_logits = causal_lm.lm_head(hs)
                past_key_values = bo.past_key_values

            logits = base_logits[:, -1, :]

            scaled_logits = logits/temperature

            values, indices = torch.topk(scaled_logits, k)
            probs = torch.softmax(values, dim=-1)

            sample = torch.multinomial(probs, 1)
            next_token_id = indices.gather(-1, sample)
            input_ids = next_token_id
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
            if x==0:
              ttfs = time.perf_counter() - start

            if next_token_id==eos:
                break

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
        total_time=time.perf_counter() - start

    elif strategy=="Top-p":
        start = time.perf_counter()
        for x in range(n):
            with torch.no_grad():
                base_output = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
                hs = base_output.last_hidden_state
                base_logits = causal_lm.lm_head(hs)
                past_key_values = base_output.past_key_values

            logits = base_logits[:, -1, :]
            scaled_logits = logits/temperature
            sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True)

            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            mask = cumulative_probs<=p
            mask[..., 0] = True

            filtered_probs = sorted_probs * mask
            filtered_probs /= filtered_probs.sum(dim=-1)

            sample = torch.multinomial(filtered_probs, num_samples=1)

            next_token_id = sorted_indices.gather(-1, sample)
            input_ids = next_token_id
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
            if x==0:
              ttfs = time.perf_counter() - start

            if next_token_id == eos:
                break

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
        total_time=time.perf_counter() - start

    else:
        print("Invalid strategy. Enter a valid strategy to generate tokens.")
    
    return {
        "total_time_taken": total_time,
        "ttfs": ttfs,
        "prompt_tokens": prompt_tokens,
        "generated_tokens": generated_ids.shape[1]-prompt_tokens
    }

## Variable Output Lengths

In [ ]:
output_lengths = [20, 100, 300, 500]

record_greedy = {}
for x in output_lengths:
    response = generate_response(
        prompt="Explain about Harry Potter and the Philosopher's Stone.",
        strategy="Greedy",
        model=qwen,
        causal_lm=causal_lm,
        tokenizer=tokenizer,
        n=x,
        eos_id=None
    )
    record_greedy[x] = response
    print()

 The Philosopher's Stone is a book written by J.K. Rowling, the author of the Harry
 The Philosopher's Stone is a book written by J.K. Rowling, the author of the Harry Potter series. It is the first book in the series and is the main source of inspiration for the characters and plot of the series. The book is a allegory for the struggle between good and evil, and the importance of knowledge and understanding. The Stone is a symbol of the power of knowledge and the ability to overcome obstacles and challenges. It is also a symbol of the importance of friendship and the
 The Philosopher's Stone is a book written by J.K. Rowling, the author of the Harry Potter series. It is the first book in the series and is the main source of inspiration for the characters and plot of the series. The book is a allegory for the struggle between good and evil, and the importance of knowledge and understanding. The Stone is a symbol of the power of knowledge and the ability to overcome obstacles and challe

In [ ]:
df = pd.DataFrame(record_greedy.items(), columns=["Output Tokens", "response"])

for x, y in df['response'].items():
    df.loc[x, 'total_time_taken'] = y['total_time_taken']
    df.loc[x, 'ttfs'] = y['ttfs']
    df.loc[x, 'prompt_tokens'] = y['prompt_tokens']

df.drop(columns=['response'], inplace=True)

df['tokens/s'] = df['Output Tokens']/df['total_time_taken']
print(df)

   Output Tokens  total_time_taken      ttfs  prompt_tokens  tokens/s
0             20          4.429964  1.437779           12.0  4.514709
1            100         14.585781  0.592511           12.0  6.855992
2            300         53.642983  0.544685           12.0  5.592530
3            500         66.647152  0.487452           12.0  7.502196


In [ ]:
prefill_ = (df["ttfs"].sum()/df["total_time_taken"].sum())
decoding_ = (df["total_time_taken"] - df["ttfs"]).sum()/df["total_time_taken"].sum()
print(f"Prefill Phase - {prefill_ * 100:.2f}%, Decoding Phase - {decoding_ * 100:.2f}%")

Prefill Phase - 2.20%, Decoding Phase - 97.80%


## Variable Prompt Lengths

In [ ]:
base_prompt = (
    "Explain the process of photosynthesis in detail, including the light-dependent "
    "reactions, the Calvin cycle, the role of chlorophyll, ATP, NADPH, carbon dioxide, "
    "glucose production, and its importance for ecosystems."
)

def make_prompt(base_prompt, tokenizer, target_tokens):
    prompt = base_prompt
    while len(tokenizer(prompt).input_ids) < target_tokens:
        prompt += " " + base_prompt
    return prompt

In [ ]:
prompt_20 = make_prompt(base_prompt, tokenizer, 20)
prompt_200 = make_prompt(base_prompt, tokenizer, 200)
prompt_500 = make_prompt(base_prompt, tokenizer, 500)
prompt_1000 = make_prompt(base_prompt, tokenizer, 1000)
prompt_2000 = make_prompt(base_prompt, tokenizer, 2000)

In [ ]:
prompts = [prompt_20, prompt_200, prompt_500, prompt_1000, prompt_2000]

record_greedy_prompt = {}
for x in prompts:
    response = generate_response(
        prompt=x,
        strategy="Greedy",
        model=qwen,
        causal_lm=causal_lm,
        tokenizer=tokenizer,
        n=150,
        eos_id=None
    )
    record_greedy_prompt[x] = response
    print()

 Provide examples of plants and animals that use photosynthesis for energy and growth. Photosynthesis is the process by which plants, algae, and some bacteria convert light energy into chemical energy in the form of glucose. The process involves several stages, including the light-dependent reactions, the Calvin cycle, and the role of chlorophyll, ATP, NADPH, carbon dioxide, glucose production, and its importance for ecosystems.

Light-dependent reactions occur in the thylakoid membranes of chloroplasts. These reactions involve the absorption of light energy by chlorophyll, which then converts it into chemical energy in the form of ATP and NADPH. The light-dependent reactions are responsible for the production of glucose, which is the primary source of energy for plants
 Explain the process of photosynthesis in detail, including the light-dependent reactions, the Calvin cycle, the role of chlorophyll, ATP, NADPH, carbon dioxide, glucose production, and its importance for ecosystems. Ex

In [ ]:
df = pd.DataFrame(record_greedy_prompt.items(), columns=["Prompt Tokens", "response"])

for x, y in df['response'].items():
    df.loc[x, 'prompt_tokens'] = y['prompt_tokens']
    df.loc[x, 'generated_tokens'] = y['generated_tokens']
    df.loc[x, 'total_time_taken'] = y['total_time_taken']
    df.loc[x, 'ttft'] = y['ttfs']

df.drop(columns=['response', 'Prompt Tokens'], inplace=True)

df['tokens/s'] = df['generated_tokens']/df['total_time_taken']
print(df)

   prompt_tokens  generated_tokens  total_time_taken        ttft  tokens/s
0           45.0             150.0         30.740676    4.081159  4.879528
1          221.0             150.0         34.799647   11.344538  4.310389
2          529.0             150.0        110.666636   21.420286  1.355422
3         1013.0             150.0        314.067488  221.882742  0.477604
4         2025.0             150.0        291.966136  258.741232  0.513758


In [ ]:
prefill_ = (df["ttft"].sum()/df["total_time_taken"].sum())
decoding_ = (df["total_time_taken"] - df["ttft"]).sum()/df["total_time_taken"].sum()
print(f"Prefill Phase - {prefill_ * 100:.2f}%, Decoding Phase - {decoding_ * 100:.2f}%")

Prefill Phase - 66.15%, Decoding Phase - 33.85%


## Temperature and P threshold

In [9]:
p_threshold = [0.2, 0.5, 0.9]
temp = [0.4, 1.2]

### Coding Test

In [ ]:
record_greedy = {}
for p in p_threshold:
    for t in temp:
        print(f"p = {p}, t = {t}\n-----------------------------------")
        response = generate_response(
            prompt="Write only a Python function that checks whether a string is a palindrome. Do not include explanations.",
            strategy="Top-p",
            model=qwen,
            causal_lm=causal_lm,
            tokenizer=tokenizer,
            p=p,
            T=t,
            n=200,
            eos_id=tokenizer.eos_token_id
        )
        record_greedy[x] = response
        print("\n")

p = 0.2, t = 0.4
-----------------------------------
 def is_palindrome(s):
    return s == s[::-1]

p = 0.2, t = 1.2
-----------------------------------
 def is_palindrome(s):
    return s == s[::-1]

p = 0.5, t = 0.4
-----------------------------------
 def is_palindrome(s):
    return s == s[::-1]

p = 0.5, t = 1.2
-----------------------------------
 Additionally, the function should also be able to handle strings with special characters and spaces. def is_palindrome(s):
    s = s.lower()
    return s == s[::-1]

p = 0.9, t = 0.4
-----------------------------------
 def is_palindrome(s):
    return s == s[::-1]

p = 0.9, t = 1.2
-----------------------------------
 #include <iostream>
#include <string>

using namespace std;

bool isPalindrome(string str) {
    string reversedStr = str;
    reverse(reversedStr.begin(), reversedStr.end());
    return str == reversedStr;
}

int main() {
    string str = "racecar";
    if (isPalindrome(str)) {
        cout << "Yes, the string is a pali

In [10]:
for p in p_threshold:
    for t in temp:
        print(f"p = {p}, t = {t}\n-----------------------------------")
        response = generate_response(
            prompt="Write a short fantasy story about a dragon who is afraid of flying.",
            strategy="Top-p",
            model=qwen,
            causal_lm=causal_lm,
            tokenizer=tokenizer,
            p=p,
            T=t,
            n=400,
            eos_id=tokenizer.eos_token_id
        )
        print("\n")

p = 0.2, t = 0.4
-----------------------------------
 Once upon a time, in a land far away, there lived a dragon named Lyra. Lyra was a fearsome creature, with wings that seemed to stretch out to infinity. She was known for her fierce beauty and her ability to soar through the air with ease.

But Lyra was not afraid of flying. She had always been fascinated by the idea of soaring through the sky, and she had always dreamed of being able to fly. She had even built a small flying machine of her own, which she had named "The Phoenix."

One day, Lyra decided to test her newfound ability by attempting to fly. She took off from the ground, and for a moment, everything seemed to be perfect. She soared through the air, soaring high above the ground, and then she began to feel a strange sense of unease. She could feel the air around her becoming thinner and thinner, and she could hear the wind whispering in her ears.

Lyra knew that she was afraid of flying, but she couldn't shake the feeling t

In [11]:
k=[3, 5, 10]
temp=[0.2, 0.6, 1.2]
record_greedy = {}
for k_ in k:
    for t in temp:
        print(f"k = {k_}, t = {t}\n-----------------------------------")
        response = generate_response(
            prompt="Write only a Python function that checks whether a string is a palindrome. Do not include explanations.",
            strategy="Top-k",
            model=qwen,
            causal_lm=causal_lm,
            tokenizer=tokenizer,
            k=k_,
            T=t,
            n=200,
            eos_id=tokenizer.eos_token_id
        )
        print("\n")

k = 3, t = 0.2
-----------------------------------
 def is_palindrome(s):
    return s == s[::-1]

k = 3, t = 0.6
-----------------------------------
 def is_palindrome(s):
    return s == s[::-1]

k = 3, t = 1.2
-----------------------------------
 The function should be able to handle strings of any length. The function should be case-sensitive. The function should return `True` if the string is a palindrome and `False` otherwise. A string is considered a palindrome if it reads the same backward as forward, ignoring spaces, punctuation, and capitalization.
To check if a string is a palindrome, you can compare the characters from both ends of the string. If all characters from both ends match, the string is a palindrome. 

Here is a Python function that checks if a string is a palindrome:

```python
def is_palindrome(s):
    s = s.replace(" ", "").lower()
    return s == s[::-1]
```

This function works as follows:
1. It replaces all spaces and punctuation marks in the string with an 